In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error

In [2]:
# Funciones de evaluación de tu paper
def paper_r2(y_true, y_pred):
    """Correlación de Pearson R"""
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return 0.0
    return np.corrcoef(y_true, y_pred)[0, 1]

def rmse(y_true, y_pred):
    """Root Mean Squared Error"""
    return np.sqrt(np.mean((y_true - y_pred)**2))

In [3]:
# 2. Función inteligente para cruzar proteínas (Incluso las que tienen ';')
def encontrar_columna_prediccion(codigo_proteina, columnas_predichas):
    # Búsqueda exacta
    if f"{codigo_proteina}_prediccion" in columnas_predichas: 
        return f"{codigo_proteina}_prediccion"
    
    # Búsqueda si hay códigos múltiples (ej. P16403;P16402)
    if pd.notna(codigo_proteina) and ';' in str(codigo_proteina):
        for sub_codigo in codigo_proteina.split(';'):
            if f"{sub_codigo}_prediccion" in columnas_predichas:
                return f"{sub_codigo}_prediccion"
    return None

In [4]:
# 1. Cargar los datasets
df_exp = pd.read_csv('datos_np_unq_rpa_externo.csv')
df_pred = pd.read_csv('datos_np_unq_externo_prediction_opt_lip-np_type.csv')
output_filename = "metricas_np_externas_opt_lip-np_type.csv"

In [5]:
# 3. MAPEO DE NANOPARTÍCULAS
# =========================================================================
# ACA DEBES PONER EL INDICE (FILA) DE DF_PRED QUE CORRESPONDE A CADA NP
# Revisa tu archivo 'datos_np_unq_externo_con_predicciones.csv' para 
# confirmar en qué fila cayó PL1, PL2, DDL1 y DDL2.
mapeo_np = {
    'PL1': 4,   # Cambia el 0 por el número de fila real de PL1
    'PL2': 5,   # Cambia el 1 por el número de fila real de PL2
    'DDL1': 6,  # Cambia el 2 por el número de fila real de DDL1
    'DDL2': 7   # Cambia el 3 por el número de fila real de DDL2
}
# =========================================================================

resultados = []

# 4. Cruzar datos y calcular métricas
for nombre_np, indice_fila in mapeo_np.items():
    valores_experimentales = []
    valores_predichos = []

    # Encuentra las proteinas para que hagan match
    for _, fila_exp in df_exp.iterrows():
        codigo = fila_exp['code']
        columna_pred = encontrar_columna_prediccion(codigo, df_pred.columns)
        
        if columna_pred is not None:
            val_real = fila_exp[nombre_np]
            val_pred = df_pred.loc[indice_fila, columna_pred]
            
            # Filtramos posibles NaNs
            if pd.notna(val_real) and pd.notna(val_pred):
                valores_experimentales.append(val_real)
                valores_predichos.append(val_pred)
                
    # 5. Cálculos
    # El Paper pide R2. A veces se refieren a Pearson al cuadrado (r^2) o al R2 Score tradicional.
    # Por las dudas, te calculo ambos.
    pearson_r = paper_r2(valores_experimentales, valores_predichos)
    pearson_r2 = pearson_r ** 2  
    error_rmse = rmse(np.array(valores_experimentales), np.array(valores_predichos))
    
    resultados.append({
        'Nanopartícula Externa': nombre_np,
        'Proteínas Cruzadas': len(valores_experimentales),
        'Pearson R': pearson_r,
        'Pearson R²': pearson_r2,
        'RMSE': error_rmse
    })

# 6. Mostrar la tabla final para responder el mail
df_resultados = pd.DataFrame(resultados)
print("=================== RESULTADOS DE VALIDACIÓN (NP EXTERNAS) ===================")
display(df_resultados) # Si usas Jupyter, 'display' imprime la tabla más bonita que 'print'

# (Opcional) Si la querés guardar para adjuntarla al mail:
df_resultados.to_csv(output_filename, index=False)

=================== RESULTADOS DE VALIDACIÓN (NP EXTERNAS) ===================


,Nanopartícula Externa,Proteínas Cruzadas,Pearson R,Pearson R²,RMSE
0,PL1,84,0.5753,0.3310,3.0259
1,PL2,84,0.4287,0.1838,2.4460
2,DDL1,84,0.2584,0.0668,2.1694
3,DDL2,84,0.2405,0.0578,2.1427
